# Understanding LLM Tokenization From Scratch
## Chapter 2: Text Preprocessing & Vocabulary Construction

Tokenization is the foundational step of building Large Language Models (LLMs). LLMs do not process raw text directly; instead, they convert sequences of characters or words into discrete numerical identifiers (token IDs) which are later mapped to vector embeddings.

### Core Objectives of this Notebook:
1. **Raw Text Acquisition & Ingestion**: Loading and inspecting the uncompressed training text.
2. **Text Preprocessing & Regex-based Splitting**: Handling word boundaries, whitespace, and punctuation preservation.
3. **Vocabulary Construction**: Mapping unique tokens to unique integer IDs and vice versa.
4. **Tokenizer V1 Implementation**: A simple bidirectional tokenizer (`encode` and `decode`).
5. **Special Context Tokens (Tokenizer V2)**: Handling Out-Of-Vocabulary (OOV) tokens (`<|unk|>`) and document boundaries (`<|endoftext|>`).

---
## 1. Raw Text Acquisition & Ingestion

We begin by loading our raw training corpus (*The Verdict* by Edith Wharton). This text represents the training corpus used to derive our tokenizer's vocabulary.

In [ ]:
with open("/home/lakshay/BuildingLargeLanguageModel/src/buildinglargelanguagemodel/data/the-verdict.txt", "r") as file:
    content = file.read()
    print(content)

---
## 2. Text Preprocessing & Token Splitting

A basic white-space split (`re.split(r'(\s)', text)`) has major limitations:
- Punctuation marks (like commas, periods, exclamation points) stick to the adjacent words (e.g., `"world!"` vs `"world"`).
- This artificially expands vocabulary size and prevents the model from understanding word semantics independently from punctuation.

To fix this, we use a regular expression that treats punctuation symbols, double dashes (`--`), and whitespace as distinct delimiter tokens.

In [76]:
import re

text = "Hello @world! This costs $50."
result = re.split(r'(\s)', text)

print(result)

['Hello', ' ', '@world!', ' ', 'This', ' ', 'costs', ' ', '$50.']


### Applying Punctuation-Aware Tokenization to the Corpus
We split on whitespace, punctuation (`[,.:;?_!"()']`), and em dashes (`--`), while filtering out empty whitespace items.

In [77]:
import re

preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', content)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])


['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


Inspect the total token count generated from the text:

In [78]:
len(preprocessed)


4690

---
## 3. Building the Vocabulary

To construct our vocabulary:
1. We extract all unique tokens using `set(preprocessed)`.
2. We sort them alphabetically to ensure deterministic token IDs.
3. We create bidirectional lookup tables:
   - `str_to_int`: Maps token string $\rightarrow$ integer ID (used in `encode`).
   - `int_to_str`: Maps integer ID $\rightarrow$ token string (used in `decode`).

In [79]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)


1130


Generate the integer-to-token dictionary:

In [ ]:
# Assign an integer to each token in the vocab
vocab = {token:integer for integer, token in enumerate(all_words)}
vocab

---
## 4. Implementing `SimpleTokenizerV1`

`SimpleTokenizerV1` implements two essential methods:
- **`encode(text)`**: Splits an input string into tokens and maps each token to its corresponding integer ID in `vocab`.
- **`decode(ids)`**: Maps integer IDs back to string tokens and cleans up whitespace before punctuation.

In [81]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

### Evaluating `SimpleTokenizerV1`
Let's test encoding and decoding a sentence from the training corpus to verify lossless round-trip reconstruction:

In [82]:
tokenizer = SimpleTokenizerV1(vocab)

text = """It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

encoded = tokenizer.encode(text)
print("encoded:", encoded)

decoded = tokenizer.decode(encoded)
print("decoded:", decoded)

encoded: [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
decoded: It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


---
## 5. Adding Special Tokens: `SimpleTokenizerV2`

### The Limitation of Tokenizer V1:
If `SimpleTokenizerV1` encounters a word or punctuation mark that was **not** present in the training corpus (such as `"ArithmeticError"` or unknown symbols), `self.str_to_int[s]` throws a `KeyError`.

### Solution:
We augment the vocabulary with special tokens:
- `<|unk|>` (Unknown): Replaces any token not found in the vocabulary during encoding.
- `<|endoftext|>`: Signifies sequence/document boundaries, allowing the LLM to process concatenated independent texts.

In [83]:
all_words = sorted(list(set(preprocessed)))
all_words.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_words)}


Inspect the updated vocabulary size and check the newly appended special tokens:

In [84]:
len(vocab.items())

1132

In [85]:
for item in list(vocab.items())[-5:]:
    print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Implementing `SimpleTokenizerV2` with Out-of-Vocabulary (OOV) Fallback
If a token does not exist in `str_to_int`, `SimpleTokenizerV2` falls back to the ID for `<|unk|>`.

In [86]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

### Evaluating `SimpleTokenizerV2`
We verify two critical scenarios:
1. Concatenating two separate texts using `<|endoftext|>`.
2. Encoding an unseen word (`ArithmeticError`) to confirm it safely maps to `<|unk|>`.

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

encoded = tokenizer.encode(text)
print("encoded:", encoded)

decoded = tokenizer.decode(encoded)
print("decoded:", decoded)

unk_word = "ArithmeticError"

encoded_unk = tokenizer.encode(unk_word)
print("encoded unk:", encoded_unk)

decoded_unk = tokenizer.decode(encoded_unk)
print("decoded unk:", decoded_unk)


Original text: Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
encoded: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
decoded: <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
encoded unk: [1131]
decoded unk: <|unk|>
